# Neural retrieval and re-ranking

Потребуются следующие данные:

- [WikIR/en1k](https://github.com/getalp/wiIR)
- [MIRAGE](https://github.com/nlpai-lab/MIRAGE)

Все данные предполагаются в директории `../data/`.

Для работы потребуется установить:

```bash
pip install sentence-transformers pytrec-eval
```

## Импорты и данные

In [1]:
import json
import os
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import pytrec_eval
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, InputExample, SentenceTransformer, util
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

d:\GitHub\itmo-ir-search\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'cuda'

### Загрузка данных

In [2]:
def load_en1k_split(base_path):
    queries = pd.read_csv(os.path.join(base_path, 'queries.csv'))
    qrels = pd.read_csv(os.path.join(base_path, 'BM25.qrels.csv'))
    bm25_res = pd.read_csv(
        os.path.join(base_path, 'BM25.res'),
        sep=' ',
        header=None,
        names=['qid', 'Q0', 'docid', 'rank', 'score', 'tag'],
    )
    return queries, qrels, bm25_res


docs = pd.read_csv('../data/en1k/documents.csv')

train_queries, train_qrels, train_bm25 = load_en1k_split('../data/en1k/training')
val_queries, val_qrels, val_bm25 = load_en1k_split('../data/en1k/validation')
test_queries, test_qrels, test_bm25 = load_en1k_split('../data/en1k/test')

print('Train:', train_queries.shape, train_qrels.shape, train_bm25.shape)
print('Validation:', val_queries.shape, val_qrels.shape, val_bm25.shape)
print('Test:', test_queries.shape, test_qrels.shape, test_bm25.shape)

with open('../data/mirage/dataset.json', 'r', encoding='utf-8') as f:
    mirage_queries_raw = json.load(f)

with open('../data/mirage/doc_pool.json', 'r', encoding='utf-8') as f:
    mirage_pool = json.load(f)

with open('../data/mirage/oracle.json', 'r', encoding='utf-8') as f:
    mirage_oracle = json.load(f)

print('Mirage queries:', len(mirage_queries_raw))
print('Mirage documents:', len(mirage_pool))
print('Mirage oracle:', len(mirage_oracle))

Train: (1444, 2) (144400, 4) (144400, 6)
Validation: (100, 2) (10100, 4) (10000, 6)
Test: (100, 2) (10100, 4) (10000, 6)
Mirage queries: 7560
Mirage documents: 37800
Mirage oracle: 7560


### Подготовка данных

In [3]:
doc_text = dict(zip(docs['id_right'], docs['text_right']))


def build_qrels_dict(df):
    qrels = defaultdict(dict)
    for row in df.itertuples():
        qrels[str(row.id_left)][str(row.id_right)] = int(row.label)
    return qrels


train_qrels_dict = build_qrels_dict(train_qrels)
val_qrels_dict = build_qrels_dict(val_qrels)
test_qrels_dict = build_qrels_dict(test_qrels)

train_query_text = dict(zip(train_queries['id_left'], train_queries['text_left']))
val_query_text = dict(zip(val_queries['id_left'], val_queries['text_left']))
test_query_text = dict(zip(test_queries['id_left'], test_queries['text_left']))

mirage_queries = []
mirage_qrels_dict = defaultdict(dict)
mirage_candidates = defaultdict(list)

for item in mirage_queries_raw:
    mirage_queries.append((item['query_id'], item['query']))

for row in mirage_pool:
    qid = row['mapped_id']
    docid = f'{qid}__{len(mirage_candidates[qid])}'
    mirage_candidates[qid].append((docid, row['doc_chunk']))
    mirage_qrels_dict[qid][docid] = int(row['support'])

mirage_query_text = dict(mirage_queries)

### Метрики и вспомогательные функции

In [4]:
def evaluate_run(run, qrels):
    metrics = {'P.1', 'P.10', 'P.20', 'map_cut.20', 'ndcg_cut.20'}
    evaluator = pytrec_eval.RelevanceEvaluator(qrels, metrics)
    scores = evaluator.evaluate(run)
    rows = pd.DataFrame(scores).T
    return {
        'p@1': rows['P_1'].mean(),
        'p@10': rows['P_10'].mean(),
        'p@20': rows['P_20'].mean(),
        'MAP@20': rows['map_cut_20'].mean(),
        'nDCG@20': rows['ndcg_cut_20'].mean(),
    }


def run_from_ranked_lists(ranked):
    run = {}
    for qid, docs_scores in ranked.items():
        run[str(qid)] = {str(docid): float(score) for docid, score in docs_scores}
    return run


def topk_dict(items, k=100):
    return sorted(items, key=lambda x: x[1], reverse=True)[:k]

## Ранжирование

In [5]:
CACHE_DIR = '../data/cache'
os.makedirs(CACHE_DIR, exist_ok=True)


def get_or_compute_doc_embeddings(model, model_alias, doc_text, batch_size=512):
    cache_path = os.path.join(CACHE_DIR, f'en1k_doc_emb_{model_alias}.pt')

    if os.path.exists(cache_path):
        data = torch.load(cache_path, map_location=DEVICE)
        return data['doc_ids'], data['doc_emb']

    doc_ids = list(doc_text.keys())
    doc_emb = model.encode(
        [doc_text[i] for i in doc_ids],
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    torch.save({'doc_ids': doc_ids, 'doc_emb': doc_emb.cpu()}, cache_path)
    return doc_ids, doc_emb


def build_mirage_cache(model, model_alias, candidates_map, batch_size=512):
    cache_path = os.path.join(CACHE_DIR, f'mirage_emb_{model_alias}.pt')
    if os.path.exists(cache_path):
        return torch.load(cache_path, map_location='cpu')

    cache = {}
    for qid, docs_local in tqdm(candidates_map.items()):
        ids_local = [x[0] for x in docs_local]
        txt_local = [x[1] for x in docs_local]
        emb = model.encode(
            txt_local,
            batch_size=batch_size,
            convert_to_tensor=True,
            normalize_embeddings=True,
        )
        cache[qid] = {'ids': ids_local, 'emb': emb.cpu()}
    torch.save(cache, cache_path)
    return cache

In [6]:
def dense_retrieval_en1k(
    model,
    model_alias,
    query_map,
    doc_text,
    batch_size=512,
    top_k=100,
):
    doc_ids, doc_emb = get_or_compute_doc_embeddings(
        model, model_alias, doc_text, batch_size
    )
    doc_emb = doc_emb.to(DEVICE)

    q_items = list(query_map.items())
    qids = [x[0] for x in q_items]

    q_emb = model.encode(
        [x[1] for x in q_items],
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    vals, inds = torch.topk(q_emb @ doc_emb.T, k=top_k, dim=1)

    ranked = {}
    for i, qid in enumerate(qids):
        ranked[qid] = [
            (doc_ids[j], float(v))
            for j, v in zip(inds[i].cpu().tolist(), vals[i].cpu().tolist())
        ]
    return ranked


def dense_retrieval_mirage(
    model,
    model_alias,
    query_map,
    candidates_map,
    batch_size=512,
):
    cache = build_mirage_cache(model, model_alias, candidates_map, batch_size)

    q_items = list(query_map.items())
    qids = [x[0] for x in q_items]
    q_emb = model.encode(
        [x[1] for x in q_items],
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    ranked = {}
    for i, qid in enumerate(tqdm(qids)):
        ids_local = cache[qid]['ids']
        doc_emb = cache[qid]['emb'].to(DEVICE)
        scores = (q_emb[i : i + 1] @ doc_emb.T)[0]
        inds = torch.argsort(scores, descending=True)
        ranked[qid] = [(ids_local[j], float(scores[j])) for j in inds.cpu().tolist()]
    return ranked


models = {
    'miniLM_L6': 'sentence-transformers/all-MiniLM-L6-v2',
    'miniLM_L12': 'sentence-transformers/all-MiniLM-L12-v2',
}

loaded_models = {
    alias: SentenceTransformer(name, device=DEVICE) for alias, name in models.items()
}

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7075.57it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7554.50it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
results = []

for alias, model in loaded_models.items():
    print(f'WikIR: {alias}')
    ranked = dense_retrieval_en1k(model, alias, test_query_text, doc_text)
    metrics = evaluate_run(run_from_ranked_lists(ranked), test_qrels_dict)
    metrics['dataset'] = 'WikIR'
    metrics['model'] = alias
    results.append(metrics)

for alias, model in loaded_models.items():
    print(f'MIRAGE: {alias}')
    ranked = dense_retrieval_mirage(model, alias, mirage_query_text, mirage_candidates)
    metrics = evaluate_run(run_from_ranked_lists(ranked), mirage_qrels_dict)
    metrics['dataset'] = 'MIRAGE'
    metrics['model'] = alias
    results.append(metrics)

pd.DataFrame(results)

WikIR: miniLM_L6


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.78it/s]


WikIR: miniLM_L12


Batches: 100%|██████████| 1/1 [00:00<00:00, 81.87it/s]


MIRAGE: miniLM_L6


100%|██████████| 7560/7560 [00:01<00:00, 4151.44it/s]


MIRAGE: miniLM_L12


100%|██████████| 7560/7560 [00:02<00:00, 3176.39it/s]


,p@1,p@10,p@20,MAP@20,nDCG@20,dataset,model
0,0.180000,0.308000,0.261500,0.036003,0.269723,WikIR,miniLM_L6
1,0.220000,0.290000,0.237000,0.030707,0.251922,WikIR,miniLM_L12
2,0.677910,0.122593,0.061296,0.800692,0.854351,MIRAGE,miniLM_L6
3,0.683995,0.122593,0.061296,0.799085,0.853543,MIRAGE,miniLM_L12


## Переранжирование

In [8]:
reranker_name = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
reranker = CrossEncoder(reranker_name, device=DEVICE)


def rerank_en1k(bm25_df, query_map, k):
    ranked = {}

    for qid, group in tqdm(bm25_df.groupby('qid')):
        qid = int(qid)
        query = query_map[qid]

        top_docs = group.sort_values('rank').head(k)['docid'].tolist()
        pairs = [(query, doc_text[d]) for d in top_docs]
        scores = reranker.predict(pairs, batch_size=64)

        reranked = list(zip(top_docs, scores))
        reranked = sorted(reranked, key=lambda x: x[1], reverse=True)
        ranked[qid] = reranked
    return ranked


for k in [10, 20, 50]:
    ranked = rerank_en1k(test_bm25, test_query_text, k)
    metrics = evaluate_run(run_from_ranked_lists(ranked), test_qrels_dict)
    metrics['dataset'] = 'WikIR'
    metrics['model'] = f'rerank_k={k}'
    results.append(metrics)


def rerank_mirage(k):
    ranked = {}

    for qid, query in tqdm(mirage_query_text.items()):
        docs_local = mirage_candidates[qid][:k]
        ids_local = [x[0] for x in docs_local]
        txt_local = [x[1] for x in docs_local]

        pairs = [(query, t) for t in txt_local]
        scores = reranker.predict(pairs, batch_size=64)

        reranked = list(zip(ids_local, scores))
        reranked = sorted(reranked, key=lambda x: x[1], reverse=True)
        ranked[qid] = reranked
    return ranked


for k in [10, 20, 50]:
    ranked = rerank_mirage(k)
    metrics = evaluate_run(run_from_ranked_lists(ranked), mirage_qrels_dict)
    metrics['dataset'] = 'MIRAGE'
    metrics['model'] = f'rerank_k={k}'
    results.append(metrics)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6448.71it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 7560/7560 [01:49<00:00, 69.21it/s]


## Смесь моделей

In [9]:
dense_model_name = models['miniLM_L12']
dense_model = loaded_models['miniLM_L12']


def minmax_norm(arr):
    arr = np.asarray(arr, dtype=float)
    if len(arr) == 0:
        return arr
    mn = arr.min()
    mx = arr.max()
    if mx == mn:
        return np.ones_like(arr)
    return (arr - mn) / (mx - mn)


def precompute_mixture_features_en1k(
    split_name,
    split_bm25,
    query_map,
    doc_text,
    model,
    batch_size=128,
):
    cache_path = os.path.join(CACHE_DIR, f'mixture_features_en1k_{split_name}.pkl')

    if os.path.exists(cache_path):
        return pd.read_pickle(cache_path)

    features = {}

    for qid, group in tqdm(
        split_bm25.groupby('qid'), desc=f'Precompute en1k {split_name}'
    ):
        qid = int(qid)
        query = query_map[qid]

        group = group.sort_values('rank')
        doc_ids = group['docid'].tolist()
        bm25_scores = minmax_norm(group['score'].values)

        texts = [doc_text[d] for d in doc_ids]

        q_emb = model.encode(
            [query],
            batch_size=1,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        d_emb = model.encode(
            texts,
            batch_size=batch_size,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        cos_scores = (q_emb @ d_emb.T)[0].detach().cpu().numpy()

        features[qid] = {
            'doc_ids': doc_ids,
            'bm25_scores': bm25_scores,
            'cos_scores': cos_scores,
        }

    pd.to_pickle(features, cache_path)
    return features


def build_run_from_mixture_features(features, alpha, top_k=100):
    ranked = {}

    for qid, item in features.items():
        final_scores = alpha * item['bm25_scores'] + (1 - alpha) * item['cos_scores']
        ranked[qid] = topk_dict(list(zip(item['doc_ids'], final_scores)), k=top_k)

    return ranked


train_mix_features = precompute_mixture_features_en1k(
    split_name='train',
    split_bm25=train_bm25,
    query_map=train_query_text,
    doc_text=doc_text,
    model=dense_model,
)

test_mix_features = precompute_mixture_features_en1k(
    split_name='test',
    split_bm25=test_bm25,
    query_map=test_query_text,
    doc_text=doc_text,
    model=dense_model,
)

best_alpha = None
best_score = -1
alpha_search = np.arange(0, 1.01, 0.05)

alpha_tuning_rows = []

for alpha in alpha_search:
    ranked = build_run_from_mixture_features(train_mix_features, alpha, top_k=100)
    metrics = evaluate_run(run_from_ranked_lists(ranked), train_qrels_dict)

    alpha_tuning_rows.append(
        {
            'alpha': alpha,
            'p@1': metrics['p@1'],
            'p@10': metrics['p@10'],
            'p@20': metrics['p@20'],
            'MAP@20': metrics['MAP@20'],
            'nDCG@20': metrics['nDCG@20'],
        }
    )

    if metrics['nDCG@20'] > best_score:
        best_score = metrics['nDCG@20']
        best_alpha = alpha

alpha_df = pd.DataFrame(alpha_tuning_rows)
alpha_df.sort_values('nDCG@20', ascending=False).head()

,alpha,p@1,p@10,p@20,MAP@20,nDCG@20
2,0.10,0.511773,0.182133,0.123996,0.336592,0.476803
3,0.15,0.503463,0.183033,0.125035,0.336073,0.475781
1,0.05,0.506925,0.179986,0.121711,0.332357,0.472435
4,0.20,0.488227,0.182410,0.125658,0.333307,0.471770
0,0.00,0.499307,0.176039,0.120083,0.325484,0.466700


In [10]:
best_alpha

np.float64(0.1)

In [11]:
ranked = build_run_from_mixture_features(test_mix_features, best_alpha, top_k=100)
metrics = evaluate_run(run_from_ranked_lists(ranked), test_qrels_dict)
metrics['dataset'] = 'WikIR'
metrics['model'] = f'mix_alpha={best_alpha:.2f}'
results.append(metrics)

### MIRAGE

In [12]:
def build_mirage_bm25_base(candidates_map, query_map):
    ranked = {}

    for qid, query in tqdm(query_map.items(), desc='BM25 for MIRAGE'):
        docs_local = candidates_map[qid]
        ids_local = [x[0] for x in docs_local]
        texts_local = [x[1] for x in docs_local]

        tokenized_docs = [text.lower().split() for text in texts_local]
        tokenized_query = query.lower().split()

        bm25 = BM25Okapi(tokenized_docs)
        scores = bm25.get_scores(tokenized_query)

        ranked[qid] = topk_dict(list(zip(ids_local, scores)), k=len(ids_local))

    return ranked


def precompute_mixture_features_mirage(
    query_map,
    candidates_map,
    model,
    batch_size=128,
):
    cache_path = os.path.join(CACHE_DIR, 'mixture_features_mirage.pkl')

    if os.path.exists(cache_path):
        return pd.read_pickle(cache_path)

    features = {}

    for qid, query in tqdm(query_map.items(), desc='Precompute MIRAGE mixture'):
        docs_local = candidates_map[qid]
        ids_local = [x[0] for x in docs_local]
        texts_local = [x[1] for x in docs_local]

        tokenized_docs = [text.lower().split() for text in texts_local]
        tokenized_query = query.lower().split()

        bm25 = BM25Okapi(tokenized_docs)
        bm25_scores = bm25.get_scores(tokenized_query)
        bm25_scores = minmax_norm(bm25_scores)

        q_emb = model.encode(
            [query],
            batch_size=1,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        d_emb = model.encode(
            texts_local,
            batch_size=batch_size,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        cos_scores = (q_emb @ d_emb.T)[0].detach().cpu().numpy()

        features[qid] = {
            'doc_ids': ids_local,
            'bm25_scores': bm25_scores,
            'cos_scores': cos_scores,
        }

    pd.to_pickle(features, cache_path)
    return features


mirage_mix_features = precompute_mixture_features_mirage(
    query_map=mirage_query_text,
    candidates_map=mirage_candidates,
    model=dense_model,
)

ranked = build_run_from_mixture_features(mirage_mix_features, best_alpha, top_k=1000)
metrics = evaluate_run(run_from_ranked_lists(ranked), mirage_qrels_dict)
metrics['dataset'] = 'MIRAGE'
metrics['model'] = f'mix_alpha={best_alpha:.2f}'
results.append(metrics)

In [13]:
mirage_bm25_ranked = build_mirage_bm25_base(mirage_candidates, mirage_query_text)

mirage_text_by_docid = {}
for qid, docs_local in mirage_candidates.items():
    mirage_text_by_docid[qid] = {docid: text for docid, text in docs_local}


def rerank_mirage_from_bm25(base_ranked, query_map, text_lookup, reranker, k):
    ranked = {}

    for qid, query in tqdm(query_map.items(), desc=f'Rerank MIRAGE k={k}'):
        top_docs = base_ranked[qid][:k]
        doc_ids = [docid for docid, _ in top_docs]
        texts = [text_lookup[qid][docid] for docid in doc_ids]

        pairs = [(query, text) for text in texts]
        scores = reranker.predict(pairs, batch_size=64)

        reranked = sorted(zip(doc_ids, scores), key=lambda x: x[1], reverse=True)
        ranked[qid] = reranked

    return ranked


mirage_rerank_rows = []

for k in [10, 20, 50]:
    ranked = rerank_mirage_from_bm25(
        base_ranked=mirage_bm25_ranked,
        query_map=mirage_query_text,
        text_lookup=mirage_text_by_docid,
        reranker=reranker,
        k=k,
    )
    metrics = evaluate_run(run_from_ranked_lists(ranked), mirage_qrels_dict)
    metrics['dataset'] = 'MIRAGE'
    metrics['model'] = f'rerank_bm25_k={k}'
    results.append(metrics)
    mirage_rerank_rows.append(metrics)

pd.DataFrame(mirage_rerank_rows)

Rerank MIRAGE k=50: 100%|██████████| 7560/7560 [01:51<00:00, 67.78it/s]


,p@1,p@10,p@20,MAP@20,nDCG@20,dataset,model
0,0.824603,0.122593,0.061296,0.889057,0.920521,MIRAGE,rerank_bm25_k=10
1,0.824603,0.122593,0.061296,0.889057,0.920521,MIRAGE,rerank_bm25_k=20
2,0.824603,0.122593,0.061296,0.889057,0.920521,MIRAGE,rerank_bm25_k=50


## Дообучение кросс-энкодера

In [14]:
def build_bm25_topn_negatives(train_bm25_df, top_n=100):
    bm25_topn = defaultdict(list)

    for qid, group in train_bm25_df.groupby('qid'):
        group = group.sort_values('rank').head(top_n)
        bm25_topn[int(qid)] = group['docid'].tolist()

    return bm25_topn


def build_crossencoder_train_examples(
    train_query_text,
    train_qrels_dict,
    bm25_topn,
    doc_text,
    negatives_per_positive=3,
):
    examples = []

    for qid_str, rels in tqdm(train_qrels_dict.items(), desc='Build train examples'):
        qid = int(qid_str)
        query = train_query_text[qid]

        positives = [int(docid) for docid, rel in rels.items() if rel > 0]
        positive_set = set(positives)

        candidate_negatives = [
            docid for docid in bm25_topn[qid] if docid not in positive_set
        ]

        if len(candidate_negatives) == 0:
            continue

        for docid in positives:
            examples.append(
                InputExample(
                    texts=[query, doc_text[docid]],
                    label=1.0,
                )
            )

            sampled_negatives = random.sample(
                candidate_negatives,
                k=min(negatives_per_positive, len(candidate_negatives)),
            )

            for neg_docid in sampled_negatives:
                examples.append(
                    InputExample(
                        texts=[query, doc_text[neg_docid]],
                        label=0.0,
                    )
                )

    return examples


train_bm25_top100 = build_bm25_topn_negatives(train_bm25, top_n=100)

train_examples = build_crossencoder_train_examples(
    train_query_text=train_query_text,
    train_qrels_dict=train_qrels_dict,
    bm25_topn=train_bm25_top100,
    doc_text=doc_text,
    negatives_per_positive=3,
)

len(train_examples)

Build train examples: 100%|██████████| 1444/1444 [00:00<00:00, 25317.27it/s]


24692

In [15]:
train_loader = DataLoader(train_examples, shuffle=True, batch_size=16)

ft_model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2',
    num_labels=1,
    device=DEVICE,
)

ft_model.fit(
    train_dataloader=train_loader,
    epochs=1,
    warmup_steps=max(1, len(train_loader) // 10),
    show_progress_bar=True,
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6324.43it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
500,0.461978
1000,0.390584
1500,0.380937


In [16]:
def rerank_en1k_with_model(model, bm25_df, query_map, doc_text, k):
    ranked = {}

    for qid, group in tqdm(bm25_df.groupby('qid'), desc=f'Rerank WikIR ft k={k}'):
        qid = int(qid)
        query = query_map[qid]

        top_docs = group.sort_values('rank').head(k)['docid'].tolist()
        pairs = [(query, doc_text[docid]) for docid in top_docs]
        scores = model.predict(pairs, batch_size=64)

        ranked[qid] = sorted(zip(top_docs, scores), key=lambda x: x[1], reverse=True)

    return ranked


ranked = rerank_en1k_with_model(
    model=ft_model,
    bm25_df=test_bm25,
    query_map=test_query_text,
    doc_text=doc_text,
    k=50,
)
metrics = evaluate_run(run_from_ranked_lists(ranked), test_qrels_dict)
metrics['dataset'] = 'WikIR'
metrics['model'] = 'ft_cross_encoder'
results.append(metrics)

Rerank WikIR ft k=50: 100%|██████████| 100/100 [00:09<00:00, 10.64it/s]


### Дообучение на MIRAGE

In [17]:
def rerank_mirage_with_model(model, base_ranked, query_map, text_lookup, k):
    ranked = {}

    for qid, query in tqdm(query_map.items(), desc=f'Rerank MIRAGE ft k={k}'):
        top_docs = base_ranked[qid][:k]
        doc_ids = [docid for docid, _ in top_docs]
        texts = [text_lookup[qid][docid] for docid in doc_ids]

        pairs = [(query, text) for text in texts]
        scores = model.predict(pairs, batch_size=64)

        ranked[qid] = sorted(zip(doc_ids, scores), key=lambda x: x[1], reverse=True)

    return ranked


ranked = rerank_mirage_with_model(
    model=ft_model,
    base_ranked=mirage_bm25_ranked,
    query_map=mirage_query_text,
    text_lookup=mirage_text_by_docid,
    k=50,
)
metrics = evaluate_run(run_from_ranked_lists(ranked), mirage_qrels_dict)
metrics['dataset'] = 'MIRAGE'
metrics['model'] = 'ft_cross_encoder'
results.append(metrics)

Rerank MIRAGE ft k=50: 100%|██████████| 7560/7560 [01:50<00:00, 68.20it/s]


In [18]:
final_df = pd.DataFrame(results)
final_df = final_df[
    ['dataset', 'model', 'p@1', 'p@10', 'p@20', 'MAP@20', 'nDCG@20']
].sort_values(['dataset', 'nDCG@20'], ascending=[True, False])

final_df

,dataset,model,p@1,p@10,p@20,MAP@20,nDCG@20
7,MIRAGE,rerank_k=10,0.824603,0.122593,0.061296,0.889057,0.920521
8,MIRAGE,rerank_k=20,0.824603,0.122593,0.061296,0.889057,0.920521
9,MIRAGE,rerank_k=50,0.824603,0.122593,0.061296,0.889057,0.920521
12,MIRAGE,rerank_bm25_k=10,0.824603,0.122593,0.061296,0.889057,0.920521
13,MIRAGE,rerank_bm25_k=20,0.824603,0.122593,0.061296,0.889057,0.920521
14,MIRAGE,rerank_bm25_k=50,0.824603,0.122593,0.061296,0.889057,0.920521
16,MIRAGE,ft_cross_encoder,0.817593,0.122593,0.061296,0.887624,0.919025
11,MIRAGE,mix_alpha=0.10,0.715212,0.122593,0.061296,0.819188,0.868550
2,MIRAGE,miniLM_L6,0.677910,0.122593,0.061296,0.800692,0.854351
3,MIRAGE,miniLM_L12,0.683995,0.122593,0.061296,0.799085,0.853543
